# Lesson 5 | Boolean logic, data selection, and clocked registers

A digital circuit calculates conditions, selects data, and stores results at specified times. Start with AND, OR, and NOT truth tables, then read a complete SystemVerilog accumulator, including every part of `always_ff @(posedge clk)`.


## 1. Concept ledger

**Boolean logic** combines false/true conditions, represented by 0/1. **Combinational logic** calculates from current inputs. A **register** stores a multi-bit state. A **positive clock edge** is the 0→1 transition of a known clock. This lesson connects the state update from Lesson 4 to concrete language constructs.


## 2. NOT, AND, OR, and truth tables

NOT inverts one condition. AND requires both to be true. OR requires at least one, including the case where both are true.

| a | b | NOT a | a AND b | a OR b |
|---|---|---|---|---|
| 0 | 0 | 1 | 0 | 0 |
| 0 | 1 | 1 | 0 | 1 |
| 1 | 0 | 0 | 0 | 1 |
| 1 | 1 | 0 | 1 | 1 |

A **truth table** lists every input combination. Python `False/True` represent 0/1 here; `not/and/or` implement these operations for Boolean operands. `product(..., repeat=2)` enumerates four pairs. Predict the `False,True` row before running.


In [ ]:
from itertools import product

def NOT(a): return not a
def AND(a, b): return a and b
def OR(a, b): return a or b

print(' a     b   NOT a   a AND b   a OR b')
for a, b in product([False, True], repeat=2):
    print(f'{a!s:5} {b!s:5} {NOT(a)!s:7} {AND(a,b)!s:9} {OR(a,b)!s:8}')


## 3. A comparator produces a logic condition

A **comparator** compares two numbers and produces a Boolean condition. `candidate_v >= threshold` includes equality. Requiring an enable gives:

`spike_condition = enable AND (candidate_v >= threshold)`

| enable | candidate_v | threshold | reached | spike_condition |
|---|---:|---:|---|---|
| 0 | 4 | 4 | 1 | 0 |
| 1 | 3 | 4 | 0 | 0 |
| 1 | 4 | 4 | 1 | 1 |

Predict the last row when `>=` becomes `>`.


In [ ]:
threshold = 4
for enable, value in [(False, 4), (True, 3), (True, 4)]:
    reached = value >= threshold
    spike_condition = enable and reached
    print(f'enable={enable}, value={value}, reached={reached}, spike_condition={spike_condition}')


## 4. A condition selects the value to store

A **multiplexer (mux)** selects one input value as its output. To store zero after a spike and the candidate otherwise:

`value_to_store = 0 if spike_condition else candidate_v`

The comparator produces `reached`, AND combines it with enable, and the mux selects data. The register stores the selected value at a clock edge. Comparison, logic operations, and selection are combinational.


## 5. Write the state update in SystemVerilog

A **Hardware Description Language (HDL)** describes digital hardware structure and behavior. **SystemVerilog** is an HDL that also supports verification. The repository accumulator clears on reset, otherwise adding the input to old state at each positive edge.

```systemverilog
module clocked_accumulator (
    input  logic              clk,
    input  logic              rst_n,
    input  logic signed [7:0] input_value,
    output logic signed [7:0] state
);
    always_ff @(posedge clk) begin
        if (!rst_n)
            state <= '0;
        else
            state <= state + input_value;
    end
endmodule
```

`module` / `endmodule` delimit a module; `clocked_accumulator` is its user-defined name. The parentheses declare **ports**: `input` and `output` specify direction. There are three inputs (`clk`, `rst_n`, `input_value`) and one output (`state`).

`logic` is a four-state data type, `signed` specifies signed interpretation, and `[7:0]` specifies eight bits. Storage is determined by the clocked assignments, not the word `logic` alone. An 8-bit two's-complement value ranges from −128 to 127; this example retains the low eight bits on overflow.


### Read `always_ff @(posedge clk)` one part at a time

**`always_ff` is a standard SystemVerilog keyword and a reserved word.** The language defines it; this is not a call to a user-defined function. `ff` refers to a flip-flop: one stores one bit, and eight can form an 8-bit register.

| Text | Language meaning | Role here |
|---|---|---|
| `always_ff` | A procedure for sequential behavior | Describes updates to `state` |
| `@(...)` | Event control: wait for the specified event | Specifies when to update |
| `posedge` | Standard keyword for a positive edge | 0→1 for a known two-state clock |
| `clk` | User-defined signal name | The module's clock input |
| `begin` / `end` | Group statements into a block | Group reset and accumulation branches |

In simulation the procedure waits for a positive clock edge, executes its statements, and waits for the next edge. In the corresponding circuit, the adder computes as the inputs and stored state change; the register samples the value at a positive edge. A high level, a low level, or a falling edge keeps `state` unchanged. Physical register outputs settle after propagation delay following the edge.

SystemVerilog `logic` also represents `X` (unknown) and `Z` (high impedance). Four-state `posedge` includes 0→X/Z and X/Z→1. The timing tables here use a known 0/1 clock.


### `if (!rst_n)`, `'0`, and `<=`

`if` and `else` are conditional-statement keywords. `!` is logical NOT; `rst_n=0` selects the reset branch. `rst_n` is a user-defined name. The `_n` suffix is an active-low naming convention; the name itself has no effect on circuit behavior.

`'0` fills the assignment destination with zeros: eight zeros for `state`. In an assignment statement, `<=` means **nonblocking assignment**; in a comparison expression it means less than or equal. Its syntax determines the meaning.

At a positive edge, `state <= state + input_value;` evaluates the right side using the old state and current input, then schedules the destination update in the nonblocking-assignment region of the current simulation time step. Old state 3 and input 2 produce stored state 5 after this edge, not one cycle later.

For `a <= b; b <= a;` in one clocked block, with no other writers, both right sides use the values before the updates, so the stored values exchange. Combinational calculations normally use blocking `=` so later statements use intermediate results already calculated.

This is a **synchronous, active-low reset**. A low `rst_n` clears `state` only at a positive clock edge. Taking reset low between edges leaves the state unchanged until the next edge. A common asynchronous form is `@(posedge clk or negedge rst_n)`, where asserting reset also triggers the procedure.


## 6. Circuit and values around an edge

The stored state and input feed the adder. Reset controls a mux selecting zero or the sum. That mux feeds the data input of an 8-bit register; `clk` feeds its clock input. The register output also feeds the adder.

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 840 300" role="img" aria-label="l05-state-update" style="width:100%;height:auto">
<defs><marker id="l05-state-update" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 Z" fill="#506070"/></marker></defs>
<rect x="35" y="90" width="150" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="110.0" y="125.0" text-anchor="middle" font-family="sans-serif" font-size="15">input_value</text>
<rect x="240" y="90" width="130" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="305.0" y="125.0" text-anchor="middle" font-family="sans-serif" font-size="15">adder</text>
<rect x="445" y="90" width="145" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="517.5" y="125.0" text-anchor="middle" font-family="sans-serif" font-size="15">reset mux</text>
<rect x="660" y="90" width="155" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="737.5" y="125.0" text-anchor="middle" font-family="sans-serif" font-size="15">state register</text>
<rect x="440" y="10" width="150" height="45" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="515.0" y="37.5" text-anchor="middle" font-family="sans-serif" font-size="15">rst_n</text>
<rect x="660" y="210" width="155" height="45" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="737.5" y="237.5" text-anchor="middle" font-family="sans-serif" font-size="15">clk</text>
<path d="M185 120 H240" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<path d="M370 120 H445" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<text x="407" y="110" text-anchor="middle" font-family="sans-serif" font-size="14">sum</text>
<path d="M590 120 H660" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<text x="622" y="110" text-anchor="middle" font-family="sans-serif" font-size="14">data</text>
<path d="M515 55 V90" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<path d="M737 210 V150" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<text x="765" y="185" text-anchor="middle" font-family="sans-serif" font-size="14">clock</text>
<path d="M815 120 H830 V285 H305 V150" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l05-state-update)"/>
<text x="470" y="280" text-anchor="middle" font-family="sans-serif" font-size="14">stored state</text>
</svg>

With old state 3 and input 2:

| Event | rst_n | state | Reason |
|---|---|---:|---|
| Before positive edge | 1 | 3 | Sum 5 is not stored yet |
| After positive edge | 1 | 5 | Sum sampled |
| Input changes to 7 | 1 | 5 | Adder changes, storage holds |
| Falling edge | 1 | 5 | No update at falling edge |
| Reset asserted between edges | 0 | 5 | Synchronous reset waits |
| After next positive edge | 0 | 0 | Reset branch clears state |


## 7. Run: check edge updates after predicting

This Python model computes values after each positive edge. Predict the three results. All sums remain within the signed 8-bit range, so Python integers and the example RTL agree.


In [ ]:
state = 0
for edge, (rst_n, input_value) in enumerate([(False, 9), (True, 2), (True, 3)]):
    before = state
    state = 0 if not rst_n else state + input_value
    print(f'edge={edge}, before={before}, rst_n={rst_n}, input={input_value}, after={state}')


### Three procedures and tool checks

Verilog-2001 already has the general `always`, such as `always @(posedge clk)`. SystemVerilog retains it and adds these standard keywords:

| Keyword | Purpose | What to inspect |
|---|---|---|
| `always_ff` | Sequential behavior such as flip-flops | Update events, reset, stored variables |
| `always_comb` | Combinational behavior | Every output assigned on every branch |
| `always_latch` | Level-sensitive latch behavior | Data passes while enabled and is held otherwise |

`always_ff` requires exactly one event control and prohibits blocking timing controls. Its assigned variables cannot also be written by another process. `@(posedge clk or negedge rst_n)` is one event control containing two events. This rule is not equivalent to requiring exactly one `posedge`.

The expression `state + input_value` inside `always_ff` still describes combinational addition. The block specifies when to store the result. Tools can check the intended sequential behavior, but diagnostics vary; every modeling mistake is not guaranteed to produce an error.

`always_comb` derives sensitivity from relevant reads and executes once at simulation startup. **Automatic sensitivity does not ensure complete assignments.** `if (enable) y = d;` leaves `y` unchanged when disabled and therefore describes latch behavior. Adding `else y = 0;` supplies a combinational result for both cases.


## 8. Try It

1. Old state is 6, input −2, reset inactive. What is stored after the next positive edge?
2. Why does changing input while keeping `clk=1` leave the register unchanged?
3. Assert synchronous reset before a falling edge. When does clearing happen?
4. Give disabled, below-threshold, and exactly-at-threshold cases for the spike condition.


## 9. Exercise

[Lesson 05 exercise: logic gates, comparator, and enable](../../exercises/en/05_logic_building_blocks.ipynb)

The exercise implements conditions. Use the timing questions above to check the added register reading material; Lesson 6 compiles this accumulator with an HDL tool.


## 10. AI Task

Ask AI to annotate the accumulator, distinguish keywords from user names, list three inputs, and tabulate before-state, reset, input, and after-state. Check the synchronous reset and old-state explanations yourself.


## 11. Human Check

Explain `always_ff`, `@`, `posedge`, `clk`, `<=`, `!rst_n`, and `'0`. Identify the adder, mux, and register. Explain why automatic sensitivity cannot supply a missing assignment.


## 12. Engineering Handoff

Lesson 6 names these relationships Register-Transfer Level (RTL) and compiles `clocked_accumulator.sv`. Lesson 7 adds threshold and spike state. Lesson 8 generates clock and input stimuli and checks outputs automatically.


## 13. Project Trace

- Lesson: `LSN-005`
- Engineering slice: `RMD-003A`
- Prepares: `RMD-004`
- Learning requirement: `PFR1 / PDP1`
- Example arithmetic: signed 8-bit wraparound; formal neuron policy belongs to `RMD-002 / RMD-003`.

Syntax reference: [SystemVerilog Language Reference Manual](https://courses.cs.washington.edu/courses/cse371/references/SystemVerilog_Reference_Manual.pdf), sections 9.2–9.4 and the keyword list.


## 14. Exit Ticket

Explain AND, OR, and NOT with truth tables. Identify combinational calculation and stored state in the accumulator. Read its clock event control and calculate both reset and accumulation branches.
